## This notebook merges the slice check CSV files of each patient into a single file.
227 patient slice_check.csv files -> one single file

In [ ]:
import pandas as pd
import os
import glob
'''
quantitative file stored path
'''
# # COIN
# quant_file_dir = '/media/NAS06/gavinyue/disentanglement/scripts_segmentation/unet_train_test/quantification_result/coin_genlabel0'
# save_path = '/media/NAS06/gavinyue/genai-wsss/experiments/wsss_coin/results/AIPFR/quant/slice_check.csv'

# # full_supervised_unet
# quant_file_dir = '/media/NAS06/gavinyue/disentanglement/scripts_segmentation/unet_train_test/quantification_result/full_supervised_unet'
# save_path = '/media/NAS06/gavinyue/genai-wsss/experiments/full_supervised_unet/results/AIPFR/quant/slice_check.csv'

# WSSS unet
quant_file_dir = '/media/NAS06/gavinyue/disentanglement/scripts_segmentation/unet_train_test/exp_img'
save_path = '/media/NAS06/gavinyue/genai-wsss/experiments/wsss_unet/results/AIPFR/quant/slice_check.csv'
postfix = '_pixel_check.csv'


In [8]:
quant_files = glob.glob(os.path.join(quant_file_dir, '*' + postfix))
quant_files.sort()
print(len(quant_files))
assert len(quant_files) == 227, 'The number of files is not correct'
pd.concat([pd.read_csv(file) for file in quant_files], axis=0).to_csv(save_path, index=False)

227


In [9]:
df = pd.read_csv(save_path)
print(df.head())
print(df.columns)

       Case                     ID     voxel  pixel_num_lung  \
0  02_00019  case02_00019_slice100  0.617981             756   
1  02_00019  case02_00019_slice101  0.617981             858   
2  02_00019  case02_00019_slice102  0.617981             972   
3  02_00019  case02_00019_slice103  0.617981            1134   
4  02_00019  case02_00019_slice104  0.617981            1158   

   pixel_num_fibrosis  slice_volume_lung  slice_volume_fibrosis  
0                 168         467.193604             103.820801  
1                 184         530.227661             113.708496  
2                 232         600.677490             143.371582  
3                 296         700.790405             182.922363  
4                 316         715.621948             195.281982  
Index(['Case', 'ID', 'voxel', 'pixel_num_lung', 'pixel_num_fibrosis',
       'slice_volume_lung', 'slice_volume_fibrosis'],
      dtype='object')


In [10]:
new_order = ['Case', 'ID','voxel','pixel_num_lung','slice_volume_lung', 'pixel_num_fibrosis', 'slice_volume_fibrosis']
df = df[new_order]
print(df.columns)

Index(['Case', 'ID', 'voxel', 'pixel_num_lung', 'slice_volume_lung',
       'pixel_num_fibrosis', 'slice_volume_fibrosis'],
      dtype='object')


In [11]:
df.insert(df.columns.get_loc('ID') + 1, 'size', 255)
print(df.head())
df.to_csv(save_path, index=False)
print('Merged Slice check quantitative csv file Done')

       Case                     ID  size     voxel  pixel_num_lung  \
0  02_00019  case02_00019_slice100   255  0.617981             756   
1  02_00019  case02_00019_slice101   255  0.617981             858   
2  02_00019  case02_00019_slice102   255  0.617981             972   
3  02_00019  case02_00019_slice103   255  0.617981            1134   
4  02_00019  case02_00019_slice104   255  0.617981            1158   

   slice_volume_lung  pixel_num_fibrosis  slice_volume_fibrosis  
0         467.193604                 168             103.820801  
1         530.227661                 184             113.708496  
2         600.677490                 232             143.371582  
3         700.790405                 296             182.922363  
4         715.621948                 316             195.281982  
Merged Slice check quantitative csv file Done


## search `real_volume_quantification_512.csv` and add size column

In [17]:
import pandas as pd
import os
import glob

csv_name = 'real_volume_quantification_512.csv'
search_path = '/media/NAS06/gavinyue/genai-wsss/experiments/**/quant/' + csv_name
print(search_path)
files = glob.glob(search_path, recursive=True)
print(files)

/media/NAS06/gavinyue/genai-wsss/experiments/**/quant/real_volume_quantification_512.csv
['/media/NAS06/gavinyue/genai-wsss/experiments/full_supervised_unet/results/AIPFR/quant/real_volume_quantification_512.csv', '/media/NAS06/gavinyue/genai-wsss/experiments/wsss_unet/results/AIPFR/quant/real_volume_quantification_512.csv', '/media/NAS06/gavinyue/genai-wsss/experiments/wsss_coin/results/AIPFR/quant/real_volume_quantification_512.csv']


In [23]:
for file in files:
    df = pd.read_csv(file)
    print(df.columns)
    if 'size' in df.columns:
        print('size already exists')
        continue
    
    df.insert(df.columns.get_loc('case') + 1, 'size', 512)
    print(df.head())
    df.to_csv(file, index=False)
    print('Update size Done')

Index(['case', 'size', 'volume_lung', 'volume_fibrosis', 'total_pixel_lung',
       'total_pixel_fibrosis'],
      dtype='object')
size already exists
Index(['case', 'size', 'volume_lung', 'volume_fibrosis', 'total_pixel_lung',
       'total_pixel_fibrosis'],
      dtype='object')
size already exists
Index(['case', 'size', 'volume_lung', 'volume_fibrosis', 'total_pixel_lung',
       'total_pixel_fibrosis'],
      dtype='object')
size already exists


### Add one column of `slice_volume_fibrosis` to `infer_slice_check.csv` 

In [3]:
import pandas as pd
infer_slice_check_csv = '/media/NAS06/gavinyue/genai-wsss/experiments/full_supervised_unet/results/CT/quant/infer_slice_check.csv'
df = pd.read_csv(infer_slice_check_csv)
print(df.columns)
if 'slice_volume_fibrosis' not in df.columns:
    df['slice_volume_fibrosis'] = df['voxel'] * df['pixel_num_fibrosis']
    print(df.head())
    df.to_csv(infer_slice_check_csv, index=False)
    print('Add slice_volume_fibrosis column Done')

Index(['Case', 'ID', 'size', 'voxel', 'pixel_num_lung', 'slice_volume_lung',
       'pixel_num_fibrosis'],
      dtype='object')
      Case                    ID  size     voxel  pixel_num_lung  \
0  0513585  case0513585_slice000   256  0.263859               0   
1  0513585  case0513585_slice001   256  0.263859              54   
2  0513585  case0513585_slice002   256  0.263859             171   
3  0513585  case0513585_slice003   256  0.263859             471   
4  0513585  case0513585_slice004   256  0.263859             654   

   slice_volume_lung  pixel_num_fibrosis  slice_volume_fibrosis  
0           0.000000                 0.0               0.000000  
1          14.248375                 0.0               0.000000  
2          45.119854               188.0              49.605453  
3         124.277493               240.0              63.326111  
4         172.563652               388.0             102.377213  
Add slice_volume_fibrosis column Done


In [ ]:
# Get summary of total fibrosis volume for all cases
import pandas as pd
import numpy as np
import os

def quantify_volumes(csv_file_path):
    '''
    Calculate the volume_lung | volume_fibrosis for each case
    '''
    default_size = 512
    quantify_path = os.path.join(os.path.dirname(csv_file_path), 'real_volume_quantification_512.csv')

    df_slice = pd.read_csv(csv_file_path)
    
    # Group by case name and process each case separately
    grouped = df_slice.groupby('Case')
    
    all_results = []
    
    for case_name, group in grouped:
        slice_volume_lung = np.array(group['slice_volume_lung'].values)
        slice_volume_fibrosis = np.array(group['slice_volume_fibrosis'].values)

        size = group['size'].values[0]  # Assuming size is same for all slices of a case
        ratio_of_real_size = default_size / size if size != 0 else 1

        volume_lung = np.sum(slice_volume_lung) * ratio_of_real_size * ratio_of_real_size
        volume_fibrosis = np.sum(slice_volume_fibrosis) * ratio_of_real_size * ratio_of_real_size

        result = {
            'case': case_name,
            'size': default_size, 
            'volume_lung': volume_lung,
            'volume_fibrosis': volume_fibrosis
        }
        all_results.append(result)

    # Create dataframe with all results and save
    df_quantify = pd.DataFrame(all_results)
    df_quantify.to_csv(quantify_path, index=False)
    print(f'All cases processed and saved to {quantify_path}')
    return df_quantify

infer_slice_check_csv = '/media/NAS06/gavinyue/genai-wsss/experiments/full_supervised_unet/results/CT/quant/infer_slice_check.csv'
df_quantify = quantify_volumes(infer_slice_check_csv)
print(df_quantify.head())

Case 0513585: volume_lung=34493368.45, volume_fibrosis=1856962.21
Case 0513666: volume_lung=45566711.82, volume_fibrosis=97795.19
Case 0514090: volume_lung=13738715.54, volume_fibrosis=661433.12
Case 0516435: volume_lung=23811684.15, volume_fibrosis=103894.68
Case 0518580: volume_lung=14695757.46, volume_fibrosis=537380.36
Case 0520477: volume_lung=24141205.99, volume_fibrosis=1194786.14
Case 0524287: volume_lung=17239739.15, volume_fibrosis=958022.19
Case 0526150: volume_lung=20775711.66, volume_fibrosis=1419882.26
Case 0527090: volume_lung=22791798.36, volume_fibrosis=1223906.23
Case 0529802: volume_lung=27197979.66, volume_fibrosis=376259.94
Case 0532860: volume_lung=30395189.86, volume_fibrosis=29015.99
Case 0600860: volume_lung=14985397.73, volume_fibrosis=549243.62
Case 0604043: volume_lung=22867218.15, volume_fibrosis=504369.08
Case 0604627: volume_lung=13169395.76, volume_fibrosis=409231.72
Case 0608418: volume_lung=8778155.82, volume_fibrosis=246939.70
Case 0612741: volume_lun